In [ ]:
import pandas as pd
import numpy as np
import tushare as ts
import matplotlib.pyplot as plt

In [ ]:
#先做一些基础设置
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

#下面是个人设置，因人而异，可修改部分均列在此
TS_TOKEN = 'your_token_here'
TS_CODE = 'your_code_here'
START_DATE = 'your_start_date_here'
END_DATE = 'your_end_date_here'

In [ ]:
#用tushare获取每日数据（当然，自己想要什么数据就用什么函数）
ts.set_token(TS_TOKEN)
pro = ts.pro_api()
data = pro.daily(ts_code=TS_CODE, start_date=START_DATE, end_date=END_DATE)
data_sorted = data.sort_values('trade_date')
print(data_sorted.head(10))

In [ ]:
#然后我们先列一下最简单的双均线策略，看看最基础的量化交易时间序列数据处理方法
data_sorted['return'] = data_sorted['close'].pct_change()
data_sorted['ma5'] = data_sorted['close'].rolling(window=5).mean()
data_sorted['ma10'] = data_sorted['close'].rolling(window=10).mean()
data_sorted.loc[:, 'signal'] = 0
data_sorted.loc[data_sorted['ma5'] > data_sorted['ma10'], 'signal'] = 1
data_sorted.loc[data_sorted['ma5'] < data_sorted['ma10'], 'signal'] = -1
print(data_sorted[['close','ma5','ma10','signal']].head(20))
data_sorted['position'] = data_sorted['signal'].diff()

data_sorted['position_shifted'] = data_sorted['signal'].shift(1)
data_sorted['strategy_return'] = data_sorted['position_shifted'] * data_sorted['return']
data_strategy = data_sorted.dropna().copy()
data_strategy['net_value'] = (1 + data_strategy['strategy_return']).cumprod()
data_strategy['benchmark'] = (1 + data_strategy['return']).cumprod()
#注意前期因为没有信号，策略净值可能会低于基准净值，这属于正常现象

In [ ]:
#绘制策略净值和基准净值的对比图
plt.figure(figsize=(12, 6))
plt.plot(data_strategy['net_value'], label='双均线策略收益', color='red', linewidth=2)
plt.plot(data_strategy['benchmark'], label='买入持有收益', color='blue', linestyle='--')
plt.title('双均线策略 vs 买入持有')
plt.xlabel('日期')
plt.ylabel('累计净值')
plt.legend()
plt.grid(True)
plt.show()